# Session 2 — Data Preparation for ML
### Hands-on notebook

**What we covered today:** the three patterns of data leakage (preprocessing, target, temporal), missing-value strategies, encoding categoricals correctly, feature scaling, and the `sklearn` Pipeline that makes leakage impossible by construction.

**What you'll do here:** build one small, realistic weekly demand dataset (styled after retail/apparel forecasting) and use it to *see* every one of today's concepts happen on real numbers — including watching a leaky feature produce a suspiciously perfect score, and watching an honest score come out worse than a dishonest one.

Run every cell top to bottom, in order. Every line of code has a comment explaining what it does.

## 1. Setup

In [1]:
# numpy: generate the synthetic data and do the underlying math
import numpy as np

# pandas: hold and manipulate our data as a table (DataFrame)
import pandas as pd

# train_test_split: split rows into a training part and a test part
from sklearn.model_selection import train_test_split

# LinearRegression, Ridge: the two models we'll train
# (Ridge adds a scale-sensitive penalty, which is why we use it for the scaling demo)
from sklearn.linear_model import LinearRegression, Ridge

# r2_score, mean_squared_error: how we'll measure prediction quality
from sklearn.metrics import r2_score, mean_squared_error

# StandardScaler: rescales a numeric column to mean 0, std 1
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# SimpleImputer: fills in missing values using a chosen strategy (median, most frequent, etc.)
from sklearn.impute import SimpleImputer

# ColumnTransformer: routes different columns down different preprocessing branches
from sklearn.compose import ColumnTransformer

# Pipeline: chains preprocessing + model into a single object
from sklearn.pipeline import Pipeline

# fix the "randomness" so everyone gets the exact same synthetic data and results
np.random.seed(7)

print("Libraries loaded.")

Libraries loaded.


## 2. Build a small synthetic demand dataset

We'll simulate 60 weeks of weekly unit sales for a jeans style, sold across 3 store regions and 2 washes — the same shape of problem as forecasting demand for a retail style-color at store level.

The data is built to deliberately contain everything Session 2 needs to practice on: missing values, categorical columns with no natural order, a genuine seasonal pattern (including one holiday-season spike), and — on purpose — one "too good to be true" feature we'll expose in a minute.

In [2]:
# how many weeks of history we're simulating, and the categories involved
n_weeks = 60
regions = ['North', 'South', 'West']
washes = ['Stonewash', 'Raw']

# made-up "true" effects each category has on demand — we use these to generate
# realistic numbers; in real life you'd never know these, the model has to find them
region_effect = {'North': 20, 'South': -10, 'West': 5}
wash_effect = {'Stonewash': 0, 'Raw': 15}

rows = []                                  # we'll collect one list per row here
start = pd.Timestamp('2024-01-01')         # the calendar date week 0 corresponds to

# loop over every week, region, and wash combination -> one row per store-week-style
for w in range(n_weeks):
    week_date = start + pd.Timedelta(weeks=w)
    # a smooth seasonal wave across the year, PLUS a holiday demand spike in the last 10 weeks
    seasonal = 15 * np.sin(2 * np.pi * w / 52) + (16 if w >= n_weeks - 10 else 0)

    for region in regions:
        for wash in washes:
            price = np.round(np.random.normal(60 if wash == 'Raw' else 50, 4), 2)
            promo = np.random.choice(['None', 'Email', 'Instore'], p=[0.6, 0.25, 0.15])
            temp = 15 + 10 * np.sin(2 * np.pi * w / 52) + np.random.normal(0, 3)

            base = 120                                  # a baseline demand level
            noise = np.random.normal(0, 6)               # random week-to-week variation
            promo_effect = {'None': 0, 'Email': 8, 'Instore': 14}[promo]

            # the "true" demand-generating formula (the model will only ever see the output)
            units = base + region_effect[region] + wash_effect[wash] - 0.6*price + promo_effect + seasonal + noise
            units = max(0, round(units))                 # demand can't be negative

            # inventory_drop is almost a copy of units_sold -- this is our planted leaky feature
            inventory_drop = units + np.random.normal(0, 2)

            rows.append([week_date, w, region, wash, price, promo, temp, units, inventory_drop])

# assemble everything into one DataFrame
df = pd.DataFrame(rows, columns=[
    'week_date', 'week_idx', 'region', 'wash', 'price',
    'promo_type', 'temperature', 'units_sold', 'inventory_drop'
])
print("Rows generated:", len(df))

Rows generated: 360


In [3]:
# now deliberately inject some missing values, like a real dataset would have

# pick 10% of rows at random and blank out temperature (e.g. a sensor outage)
miss_temp_idx = np.random.choice(df.index, size=int(0.10 * len(df)), replace=False)
df.loc[miss_temp_idx, 'temperature'] = np.nan

# pick 8% of rows at random and blank out promo_type (e.g. a logging gap)
miss_promo_idx = np.random.choice(df.index, size=int(0.08 * len(df)), replace=False)
df.loc[miss_promo_idx, 'promo_type'] = np.nan

print("Missing temperature:", df['temperature'].isna().sum())
print("Missing promo_type:", df['promo_type'].isna().sum())

Missing temperature: 36
Missing promo_type: 28


In [4]:
# always look at the data before touching it
print("Shape:", df.shape)
df.head()

Shape: (360, 9)


,week_date,week_idx,region,wash,price,promo_type,temperature,units_sold,inventory_drop
0,2024-01-01,0,North,Stonewash,56.76,Instore,13.602188,121,127.401829
1,2024-01-01,0,North,Raw,59.37,None,14.129606,125,126.200997
2,2024-01-01,0,South,Stonewash,47.50,None,14.485355,81,81.042781
3,2024-01-01,0,South,Raw,63.00,None,14.750593,91,91.247762
4,2024-01-01,0,West,Stonewash,51.10,Instore,10.420426,109,106.785765


## 3. Leakage demo 1 — the feature that's too good to be true

Recall the diagnostic heuristic: if one feature alone gives near-perfect performance, suspect leakage before you celebrate. `inventory_drop` is our planted example — it's almost a restatement of `units_sold`, the exact thing we're trying to predict. Let's watch what including it does to the score.

In [5]:
# build a clean, encoded feature table we can reuse for a few demos below.
# .fillna(median) handles the missing numeric values just so this cell runs standalone --
# we'll cover imputation properly, on purpose, in section 6.
numeric_base = ['price', 'temperature', 'week_idx']
work = df.copy()
work[numeric_base] = work[numeric_base].fillna(work[numeric_base].median())
work['promo_type'] = work['promo_type'].fillna(work['promo_type'].mode()[0])

# one-hot encode the categorical columns (drop_first avoids the dummy-variable trap)
encoded = pd.get_dummies(work, columns=['region', 'wash', 'promo_type'], drop_first=True)
category_cols = [c for c in encoded.columns if c.startswith(('region_', 'wash_', 'promo_type_'))]

y = encoded['units_sold']

# WITH the leaky feature included
X_with_leak = encoded[numeric_base + category_cols + ['inventory_drop']]
X_train, X_test, y_train, y_test = train_test_split(X_with_leak, y, test_size=0.25, random_state=42)
model_leaky = LinearRegression().fit(X_train, y_train)
r2_leaky = r2_score(y_test, model_leaky.predict(X_test))

print(f"WITH inventory_drop (leaky)     -> test R^2 = {r2_leaky:.3f}")

WITH inventory_drop (leaky)     -> test R^2 = 0.991


In [6]:
# WITHOUT the leaky feature -- same rows, same split, just one column removed
X_clean = encoded[numeric_base + category_cols]
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_clean, y, test_size=0.25, random_state=42)
model_clean = LinearRegression().fit(X_train2, y_train2)
r2_clean = r2_score(y_test2, model_clean.predict(X_test2))

print(f"WITHOUT inventory_drop (honest) -> test R^2 = {r2_clean:.3f}")
print()
print("The 'leaky' model looks almost perfect. It isn't smarter -- it was handed a")
print("disguised copy of the answer. The honest model is worse-looking and more real.")

WITHOUT inventory_drop (honest) -> test R^2 = 0.687

The 'leaky' model looks almost perfect. It isn't smarter -- it was handed a
disguised copy of the answer. The honest model is worse-looking and more real.


## 4. Leakage demo 2 — scale before split vs. after

This is the classic mistake: fitting a scaler's mean/std on the *whole* dataset before splitting, instead of fitting it on the training rows only. We use `Ridge` here (not plain linear regression) because Ridge's penalty is scale-sensitive, so this is a fair test of whether the leak actually changes anything.

In [7]:
numeric_cols = ['price', 'temperature', 'week_idx']
Xb = df[numeric_cols].fillna(df[numeric_cols].median())
yb = df['units_sold']

# WRONG: fit the scaler on the ENTIRE dataset, THEN split into train/test
scaler_wrong = StandardScaler().fit(Xb)
Xb_scaled_wrong = scaler_wrong.transform(Xb)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(Xb_scaled_wrong, yb, test_size=0.4, random_state=42)
model_wrong = Ridge(alpha=20).fit(Xb_train, yb_train)
r2_wrong = r2_score(yb_test, model_wrong.predict(Xb_test))

print(f"WRONG (scale full data, then split) -> test R^2 = {r2_wrong:.4f}")

WRONG (scale full data, then split) -> test R^2 = 0.2299


In [8]:
# CORRECT: split FIRST, then fit the scaler using the training rows only
Xb_train_raw, Xb_test_raw, yb_train2, yb_test2 = train_test_split(Xb, yb, test_size=0.4, random_state=42)
scaler_ok = StandardScaler().fit(Xb_train_raw)                 # fit on TRAIN only
Xb_train_ok = scaler_ok.transform(Xb_train_raw)                # apply to train
Xb_test_ok = scaler_ok.transform(Xb_test_raw)                  # apply the SAME scaler to test
model_ok = Ridge(alpha=20).fit(Xb_train_ok, yb_train2)
r2_ok = r2_score(yb_test2, model_ok.predict(Xb_test_ok))

print(f"CORRECT (split, then scale train only) -> test R^2 = {r2_ok:.4f}")
print()
print("Notice the gap here is tiny. That's expected -- with a large-enough, evenly")
print("mixed random split, the mean/std computed on 80-100% of the data barely differ.")
print("The discipline still isn't optional though: watch what happens next, when the")
print("test period isn't just a random slice, but a real slice of the FUTURE.")

CORRECT (split, then scale train only) -> test R^2 = 0.2297

Notice the gap here is tiny. That's expected -- with a large-enough, evenly
mixed random split, the mean/std computed on 80-100% of the data barely differ.
The discipline still isn't optional though: watch what happens next, when the
test period isn't just a random slice, but a real slice of the FUTURE.


## 5. Leakage demo 3 — random split vs. time-based split

This is temporal leakage. A random split can put a "future" week into training and a "past" week into testing -- something that can never happen in real deployment, because you always forecast forward, never backward. Same data, same model -- only the split changes.

In [ ]:
Xc = df[['price', 'temperature', 'week_idx']].fillna(df[['price', 'temperature', 'week_idx']].median())
yc = df['units_sold']

# WRONG-shaped for forecasting: shuffle every week randomly into train/test
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42)
model_random = LinearRegression().fit(Xc_train, yc_train)
r2_random = r2_score(yc_test, model_random.predict(Xc_test))

print(f"RANDOM split       -> test R^2 = {r2_random:.3f}")

In [ ]:
# CORRECT for forecasting: train on the earlier weeks, test on the later weeks --
# this mirrors exactly what "predict October using only data up to September" means.
cutoff = df['week_idx'].quantile(0.75)          # last 25% of weeks become the test period
train_mask = df['week_idx'] <= cutoff

Xc_train_t, yc_train_t = Xc[train_mask], yc[train_mask]
Xc_test_t, yc_test_t = Xc[~train_mask], yc[~train_mask]
model_time = LinearRegression().fit(Xc_train_t, yc_train_t)
r2_time = r2_score(yc_test_t, model_time.predict(Xc_test_t))

print(f"TIME-based split    -> test R^2 = {r2_time:.3f}")
print(f"(train = weeks 0-{int(cutoff)}, test = weeks {int(cutoff)+1}-{n_weeks-1})")
print()
print("The random split looks fine because some 'holiday spike' weeks leaked into")
print("training. The time-based split is worse -- and it's telling the truth: our")
print("model has never once seen a holiday spike before, because we only have one")
print("year of history. That's a real, honest limitation -- not a bug.")

## 6. Missing values — decide, don't default

Two decisions: what to fill missing values with, and whether to flag that you filled them.

In [ ]:
df2 = df.copy()

# numeric: fill with the MEDIAN (resists being dragged by a few extreme prices/temps)
temp_median = df2['temperature'].median()
df2['temperature_was_missing'] = df2['temperature'].isna().astype(int)   # flag BEFORE filling
df2['temperature'] = df2['temperature'].fillna(temp_median)

# categorical: fill with the MODE (the most common category)
promo_mode = df2['promo_type'].mode()[0]
df2['promo_was_missing'] = df2['promo_type'].isna().astype(int)          # flag BEFORE filling
df2['promo_type'] = df2['promo_type'].fillna(promo_mode)

print("temperature filled with median:", round(temp_median, 2))
print("promo_type filled with mode:", promo_mode)
print("Remaining missing values:\n", df2[['temperature', 'promo_type']].isna().sum())

## 7. Encoding categoricals — one-hot, done right

In [ ]:
# the NOMINAL TRAP, on purpose: label-encoding a category with no real order.
# this is illustrative only -- we do NOT use this encoding anywhere else.
bad_map = {'North': 0, 'South': 1, 'West': 2}
demo = df2[['region']].copy()
demo['region_label_encoded'] = demo['region'].map(bad_map)
print(demo.head())
print()
print("A linear model would now read 'West' as literally double 'South' -- there is")
print("no such relationship between regions. This is exactly why we one-hot instead.")

In [ ]:
# the CORRECT way: one-hot encode, and drop_first=True to avoid the dummy-variable trap
# (k one-hot columns + an intercept are perfectly collinear -- dropping one column fixes it)
df_encoded = pd.get_dummies(df2, columns=['region', 'wash', 'promo_type'], drop_first=True)

new_cols = [c for c in df_encoded.columns if c.startswith(('region_', 'wash_', 'promo_type_'))]
print("New one-hot columns created:", new_cols)
df_encoded[new_cols].head()

## 8. The sklearn Pipeline — every safeguard, one object

We now put imputation, encoding, and scaling into a single `Pipeline`, fit it on the training weeks only, and evaluate honestly on the later, unseen weeks. `pipeline.fit(X_train)` touches only the training fold for *every* step automatically -- there is no code path left that can leak the test rows in.

In [ ]:
feature_cols = ['price', 'temperature', 'week_idx', 'region', 'wash', 'promo_type']
X = df[feature_cols]
y = df['units_sold']

# same honest, time-based split as demo 3
cutoff = df['week_idx'].quantile(0.75)
train_mask = df['week_idx'] <= cutoff
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]

numeric_features = ['price', 'temperature', 'week_idx']
categorical_features = ['region', 'wash', 'promo_type']

# numeric branch: fill missing with median, then scale
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

# categorical branch: fill missing with the most frequent category, then one-hot encode
# handle_unknown='ignore' means a category never seen in training won't crash the pipeline
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
])

# route each column list down its matching branch, then rejoin
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features),
])

# chain preprocessing + model into ONE object
full_pipeline = Pipeline([
    ('pre', preprocessor),
    ('model', Ridge(alpha=5)),
])

# this single .fit() call fits the imputer, scaler, encoder, AND model -- all on X_train only
full_pipeline.fit(X_train, y_train)

In [ ]:
train_pred = full_pipeline.predict(X_train)
test_pred = full_pipeline.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print(f"{'Metric':<10}{'Train':>10}{'Test':>10}")
print(f"{'R^2':<10}{train_r2:>10.3f}{test_r2:>10.3f}")
print(f"{'RMSE':<10}{train_rmse:>10.2f}{test_rmse:>10.2f}")
print()
print("Every safeguard is in place, and the test score is still weak. That's not a")
print("Pipeline problem -- it's an honest data problem: our history covers only one")
print("holiday season, and it fell entirely in the test weeks. This is exactly the")
print("session's 'one to ponder' question: would more years of history fix this?")

## 9. 🔧 Try it yourself

Edit the cell below and re-run:

1. Change `test_size` from `0.75` to `0.6` in the quantile cutoff -- giving the model a bit of the holiday period to train on. Does the test score improve?
2. Swap `Ridge(alpha=5)` for `LinearRegression()`. Does the honest score change much?
3. Add `'inventory_drop'` back into `feature_cols` and re-run. Watch the score jump -- you've reintroduced the leak from section 3, this time inside the Pipeline.

In [ ]:
# 🔧 Your turn -- edit the value below and re-run
cutoff_quantile = 0.75      # try 0.6

cutoff2 = df['week_idx'].quantile(cutoff_quantile)
train_mask2 = df['week_idx'] <= cutoff2
X_train2, y_train2 = X[train_mask2], y[train_mask2]
X_test2, y_test2 = X[~train_mask2], y[~train_mask2]

full_pipeline.fit(X_train2, y_train2)
print("Test R^2 with cutoff_quantile =", cutoff_quantile, "->",
      round(r2_score(y_test2, full_pipeline.predict(X_test2)), 3))

## 10. Check yourself — five questions

Answer these in your own words before next session:

1. Name the three leakage patterns and give an example of each.
2. When do you prefer median over mean imputation?
3. Why is label-encoding a nominal feature dangerous for a linear model?
4. Which models are exempt from scaling, and why?
5. How does a Pipeline make leakage impossible?

## Recap

**Three you learned:** the three leakage patterns, made concrete on real numbers · MCAR/MAR/MNAR-style missingness decisions · encoding and scaling choices that don't invent false structure.

**Two to remember:** fit every transformer on train only · the Pipeline is what enforces that, so you don't have to trust your own memory under deadline pressure.

**One to ponder:** our honest pipeline score was weak because one year of history has only one holiday season. If more data always shrinks the train-test gap, is there ever a reason not to collect more?